# organic_ratio — entry notebook


## 1. Git sync

Подтянуть актуальный `main` из `Anton-Filimoncev-azur/organic_ratio` поверх локальной копии. Требует `GIT_PAT` в `.env`.

In [1]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "Anton-Filimoncev-azur"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio
HEAD is now at 547d1b4  work
HEAD is now at 547d1b4  work
Repository synced successfully.


From https://github.com/Anton-Filimoncev-azur/organic_ratio
 * branch            main       -> FETCH_HEAD


## 2. Env

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [7]:
%run run.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
{'project': {'name': 'kclash', 'predict_target': 'organic', 'max_horizon': 7, 'horizons': [7], 'learn_days_min': 7, 'learn_days_max': 7, 'root': '.'}, 's3': {'bucket': 'az-jupyterhub-share-ik6ahsi9', 'root': 'azur_ml_core/kclash/partition'}, 'paths': {'data': './data', 'raw': './data/raw', 'features': './data/features'}, 'datasets': {'installs': {'type': 'parquet', 's3_path': 's3://az-jupyterhub-share-ik6ahsi9/azur_ml_core/kclash/partition/installs.parquet', 'local_raw_dir': './data/raw/partition', 'local_feature_dir': './data/features/partition', 'filename': 'installs.parquet'}, 'ads': {'type': 'parquet', 's3_path': 's3://az-jupyterhub-share-ik6ahsi9/azur_ml_core/kclash/partition/ads.parquet', 'local_raw_dir': './data/raw/partition', 'local_feature_dir': './data/features/partition', 'filename': 'ads.parquet'}, 'iap': {'type': 'parquet', 's3_path': 's3://az-jupyterhub-share-ik6ahsi9/azur_ml_core/kclash/partition/iap.parquet', 'local

## 4. Per-source preprocessing (raw → data/features/partition/)

In [3]:
import gc
%run run_preprocessing.py
gc.collect()

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src

Processing dataset: installs
Saved features for installs: data/features/partition/installs.parquet

Processing dataset: ads


/home/jovyan/KEDRO/organic_ratio/src/organic_ratio/core/preprocessing/ads.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .pivot_table(


Saved features for ads: data/features/partition/ads.parquet

Processing dataset: iap


/home/jovyan/KEDRO/organic_ratio/src/organic_ratio/core/preprocessing/iap.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .pivot_table(
/home/jovyan/KEDRO/organic_ratio/src/organic_ratio/core/preprocessing/iap.py:245: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final["iap_rev_d1_3"] = df_final["iap_rev_d1_3"].fillna(0.0)
/home/jovyan/KEDRO/organic_ratio/src/organic_ratio/core/preprocessing/iap.py:247: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_ob

Saved features for iap: data/features/partition/iap.parquet

Processing dataset: devices
Saved features for devices: data/features/partition/devices.parquet

Processing dataset: costs
Saved features for costs: data/features/partition/costs.parquet

Processing dataset: sessions
[sessions preproc] raw sessions_X spans days 0..7; target max_horizon-1 = 6
Saved features for sessions: data/features/partition/sessions.parquet

Processing dataset: personal
Saved features for personal: data/features/partition/personal.parquet


717

## 5. Cohort aggregation

`run_cohort_aggregation.py` — мерджит per-source user-grain фичи и роллапит до cohort-grain по ключам из `cohort.keys`. Добавляет `cohort_size` и `n_calendar_days`.

In [3]:
%run run_cohort_aggregation.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src


TypeError: 'method' object is not iterable

## 6. Train / Eval / Inference (TODO)

PyMC иерархическая Beta-регрессия для cohort-level organic share.

In [ ]:
# %run run_train.py
# %run run_eval.py
# %run run_inference.py

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [ ]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)